In [ ]:
# Using pre-compiled wheels for CUDA 12.1+ to save time
!pip install llama-cpp-python[server] \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu122
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 GB 799.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.4 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import subprocess
import threading

server = subprocess.Popen(
    [
        "python3", "-m", "llama_cpp.server",
        "--model", "/content/drive/MyDrive/PsychoCounsel-Llama3-8B-Q4_K_M.gguf",
        "--host", "0.0.0.0",
        "--port", "8000",
        "--n_gpu_layers", "-1",
        "--n_ctx", "2048",
        "--chat_format", "chatml"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

def stream_logs():
    for line in server.stdout:
        print(line, end="")

threading.Thread(target=stream_logs).start()

In [ ]:
import requests
import time

def wait_for_server(url, timeout=60):
    print(f'Waiting for server at {url}...')
    start_time = time.time()
    while time.time() - start_time < timeout:
        try:
            res = requests.get(url)
            if res.status_code == 200:
                print('Server running ✅')
                return True
        except requests.ConnectionError:
            pass
        time.sleep(2)
    print('Server not started ❌')
    return False

# Check if server is up
if not wait_for_server('http://localhost:8000/docs'):
    print('\nCheck if the model path is correct and llama-cpp-python is installed correctly.')

Waiting for server at http://localhost:8000/docs...
INFO:     127.0.0.1:55004 - "GET /docs HTTP/1.1" 200 OK
Server running ✅


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [ ]:
import subprocess, time

# Start cloudflared tunnel
tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

print("Generating tunnel URL...")
for line in iter(tunnel.stdout.readline, b''):
    line = line.decode()
    if "trycloudflare.com" in line and "https://" in line:
        # Extract the URL from the log line
        url = [word for word in line.split() if "trycloudflare.com" in word][0]
        print(f"\n✅ YOUR API URL: {url}")
        break

Generating tunnel URL...

✅ YOUR API URL: https://setup-courtesy-virginia-cosmetic.trycloudflare.com
